# Features extraction code

## Requirements

In [ ]:
!pip3 install pandas openpyxl numpy opencv-python ollama opensmile torch torchaudio torchcodec

## Imports

In [ ]:
import pandas as pd
import openpyxl
import json
import statistics
import numpy as np
import cv2
import base64
import os
os.environ["OLLAMA_HOST"] = os.getenv("OLLAMA_HOST")
import ollama
from collections import Counter
import re
import opensmile
import torch
import torchaudio

## Dataframe creation and general variables

In [ ]:
features_df = pd.read_excel("../extracted_features/extracted_features_questionnaire.xlsx")

data_path = "../data/"
valid_participants = features_df["participant_id"].unique()

features_df

## General features

### Activity time

In [ ]:
activity_time_list = []

for participant in valid_participants:
    for condition in ["c1", "c2"]:
        stroke_file_path = data_path + "p" + str(participant) + "/drawing_data/strokes_" + condition + ".json"
        with open(stroke_file_path, "r") as file:
            data = json.load(file)
        activity_time_list.append(data["activity_time"])

features_df["activity_time"] = activity_time_list

## Drawing dynamics features

### Total stroke count

In [ ]:
total_stroke_count_list = []

for participant in valid_participants:
    for condition in ["c1", "c2"]:
        stroke_file_path = data_path + "p" + str(participant) + "/drawing_data/strokes_" + condition + ".json"
        with open(stroke_file_path, "r") as file:
            data = json.load(file)
        total_stroke_count_list.append(len(data["strokes"]) + len(data["undone_erased_strokes"]))

features_df["total_stroke_count"] = total_stroke_count_list

### Stroke length, speed, acceleration

In [ ]:
def compute_motion_features(points, timestamps):

    points = np.array(points)
    t = np.array(timestamps)

    x = points[:, 0]
    y = points[:, 1]

    dx = np.diff(x)
    dy = np.diff(y)
    dt = np.diff(t)

    # avoid division by zero
    dt[dt == 0] = np.nan

    # distance between consecutive points
    distance = np.sqrt(dx**2 + dy**2)

    # stroke length
    stroke_length = np.nansum(distance)

    # speed
    speed = distance / dt
    stroke_speed = np.nanmean(speed)

    # acceleration (only if computable)
    if len(speed) >= 2:
        acceleration = np.diff(speed) / dt[1:]
        stroke_accel = np.nanmean(acceleration)
    else:
        stroke_accel = np.nan

    return stroke_length, stroke_speed, stroke_accel

In [ ]:
stroke_total_length_list = []
stroke_mean_length_list = []
stroke_std_length_list = []
stroke_mean_speed_list = []
stroke_std_speed_list = []
stroke_mean_accel_list = []
stroke_std_accel_list = []

for participant in valid_participants:
    for condition in ["c1", "c2"]:
        stroke_file_path = data_path + "p" + str(participant) + "/drawing_data/strokes_" + condition + ".json"
        with open(stroke_file_path, "r") as file:
            data = json.load(file)

        # Extract length of each stroke of participant
        stroke_length_array = []
        stroke_speed_array = []
        stroke_accel_array = []
        for stroke in (data["strokes"] + data["undone_erased_strokes"]):
            stroke_length, stroke_speed, stroke_accel = compute_motion_features(stroke["points"], stroke["timestamps"])
            stroke_length_array.append(stroke_length)
            stroke_speed_array.append(stroke_speed)
            stroke_accel_array.append(stroke_accel)

        # Compute total stroke length, and mean/std of stroke length, speed, acceleration
        stroke_total_length_list.append(np.nansum(stroke_length_array))
        stroke_mean_length_list.append(np.nanmean(stroke_length_array))
        stroke_std_length_list.append(np.nanstd(stroke_length_array))
        stroke_mean_speed_list.append(np.nanmean(stroke_speed_array))
        stroke_std_speed_list.append(np.nanstd(stroke_speed_array))
        stroke_mean_accel_list.append(np.nanmean(stroke_accel_array))
        stroke_std_accel_list.append(np.nanstd(stroke_accel_array))

features_df["stroke_total_length"] = stroke_total_length_list
features_df["stroke_mean_length"] = stroke_mean_length_list
features_df["stroke_std_length"] = stroke_std_length_list
features_df["stroke_mean_speed"] = stroke_mean_speed_list
features_df["stroke_std_speed"] = stroke_std_speed_list
features_df["stroke_mean_accel"] = stroke_mean_accel_list
features_df["stroke_std_accel"] = stroke_std_accel_list

### Stroke pressure (mean, std, slope over time)

In [ ]:
stroke_mean_pressure_list = []
stroke_std_pressure_list = []
stroke_pressure_slope_list = []

for participant in valid_participants:
    for condition in ["c1", "c2"]:
        stroke_file_path = data_path + "p" + str(participant) + "/drawing_data/strokes_" + condition + ".json"
        with open(stroke_file_path, "r") as file:
            data = json.load(file)

        stroke_pressure_array = [np.nanmean(stroke["pressure_values"]) for stroke in data["strokes"] + data["undone_erased_strokes"]]
        stroke_pressure_array = np.array(stroke_pressure_array)
        
        # mean / std
        stroke_mean_pressure_list.append(np.nanmean(stroke_pressure_array))
        stroke_std_pressure_list.append(np.nanstd(stroke_pressure_array))

        # artificial time axis
        t = np.arange(len(stroke_pressure_array))

        # slope via linear regression
        if len(stroke_pressure_array) >= 2:
            slope, _ = np.polyfit(t, stroke_pressure_array, 1)
        else:
            slope = np.nan

        stroke_pressure_slope_list.append(slope)

features_df["stroke_mean_pressure"] = stroke_mean_pressure_list
features_df["stroke_std_pressure"] = stroke_std_pressure_list
features_df["stroke_pressure_slope"] = stroke_pressure_slope_list

### Stroke width

In [ ]:
pen_size_used_count_list = []
mean_pen_size_list = []
std_pen_size_list = []

for participant in valid_participants:
    for condition in ["c1", "c2"]:
        stroke_file_path = data_path + "p" + str(participant) + "/drawing_data/strokes_" + condition + ".json"
        with open(stroke_file_path, "r") as file:
            data = json.load(file)
        
        pen_size_array = [stroke["pen_size"] for stroke in (data["strokes"] + data["undone_erased_strokes"])]

        pen_size_used_count_list.append(len(set(pen_size_array)))
        mean_pen_size_list.append(np.mean(pen_size_array))
        std_pen_size_list.append(np.std(pen_size_array))

features_df["pen_size_used_count"] = pen_size_used_count_list
features_df["pen_size_mean"] = mean_pen_size_list
features_df["pen_size_std"] = std_pen_size_list

### Stroke complexity

In [ ]:
def compute_stroke_complexity(points):

    points = np.array(points)

    x = points[:, 0]
    y = points[:, 1]

    dx = np.diff(x)
    dy = np.diff(y)

    # Compute stroke length for later normalization
    segment_lengths = np.sqrt(dx**2 + dy**2)
    stroke_length = np.sum(segment_lengths)

    # compute turning angles
    angles = []

    for i in range(len(dx) - 1):

        v1 = np.array([dx[i], dy[i]])
        v2 = np.array([dx[i+1], dy[i+1]])

        norm1 = np.linalg.norm(v1)
        norm2 = np.linalg.norm(v2)

        if norm1 == 0 or norm2 == 0:
            continue

        cos_angle = np.dot(v1, v2) / (norm1 * norm2)
        cos_angle = np.clip(cos_angle, -1, 1)

        angle = np.arccos(cos_angle)
        angles.append(angle)

    angles = np.array(angles)

    avg_curvature = np.nanmean(angles) if len(angles) > 0 else np.nan
    angularity_index = np.nansum(angles) / stroke_length if stroke_length > 0 else np.nan

    return avg_curvature, angularity_index

In [ ]:
stroke_curvature_list = []
stroke_angularity_list = []

for participant in valid_participants:
    for condition in ["c1", "c2"]:
        stroke_file_path = data_path + "p" + str(participant) + "/drawing_data/strokes_" + condition + ".json"
        with open(stroke_file_path, "r") as file:
            data = json.load(file)

        curvature_values = []
        angularity_values = []
        for stroke in (data["strokes"] + data["undone_erased_strokes"]):
            curvature, angularity = compute_stroke_complexity(stroke["points"])
            curvature_values.append(curvature)
            angularity_values.append(angularity)

        stroke_curvature_list.append(np.nanmean(curvature_values))
        stroke_angularity_list.append(np.nanmean(angularity_values))

features_df["stroke_mean_curvature"] = stroke_curvature_list
features_df["stroke_mean_angularity"] = stroke_angularity_list

### Temporal drawing behavior

In [ ]:
avg_time_between_strokes_list = []
std_time_between_strokes_list = []
total_pause_time_list = []

for participant in valid_participants:
    for condition in ["c1", "c2"]:
        stroke_file_path = data_path + "p" + str(participant) + "/drawing_data/strokes_" + condition + ".json"
        with open(stroke_file_path, "r") as file:
            data = json.load(file)

        strokes = data["strokes"] + data["undone_erased_strokes"]

        # get start and end times of each stroke
        start_times = [stroke["timestamps"][0] for stroke in strokes]
        end_times = [stroke["timestamps"][-1] for stroke in strokes]

        # compute pauses between strokes
        pauses = []
        for i in range(len(strokes) - 1):
            pause = start_times[i+1] - end_times[i]
            pauses.append(pause)

        pauses = np.array(pauses)

        total_pause = np.nansum(pauses)
        avg_pause = np.nanmean(pauses)
        std_pause = np.nanstd(pauses)

        total_pause_time_list.append(total_pause)
        avg_time_between_strokes_list.append(avg_pause)
        std_time_between_strokes_list.append(std_pause)

features_df["avg_time_between_strokes"] = avg_time_between_strokes_list
features_df["std_time_between_strokes"] = std_time_between_strokes_list
features_df["total_pause_time"] = total_pause_time_list
features_df["pause_ratio"] = features_df["total_pause_time"] / features_df["activity_time"]
features_df["stroke_frequency"] = features_df["total_stroke_count"] / features_df["activity_time"]

### Editing behavior

In [ ]:
net_erased_pixels_list = []
undone_erased_stroke_count_list = []

for participant in valid_participants:
    for condition in ["c1", "c2"]:
        stroke_file_path = data_path + "p" + str(participant) + "/drawing_data/strokes_" + condition + ".json"
        with open(stroke_file_path, "r") as file:
            data = json.load(file)
        net_erased_pixels_list.append(data["net_erased_pixels"])
        undone_erased_stroke_count_list.append(len(data["undone_erased_strokes"]))

features_df["net_erased_pixels"] = net_erased_pixels_list
features_df["undone_erased_stroke_count"] = undone_erased_stroke_count_list
features_df["undone_erased_stroke_ratio"] = features_df["undone_erased_stroke_count"] / features_df["total_stroke_count"]

## Drawing production features

### Spatial coverage and dispersion

In [ ]:
def extract_spatial_features(image):

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # drawing mask (non-white pixels)
    mask = np.any(image != [255,255,255], axis=2)

    H, W = mask.shape

    # coordinates of non-white pixels
    ys, xs = np.where(mask)

    # no drawing case
    if len(xs) == 0:
        return [np.nan]*6

    # Coverage ratio
    coverage_ratio = len(xs) / (H * W)

    # Horizontal dispersion
    pixel_std_x = np.std(xs) / W

    # Vertical dispersion
    pixel_std_y = np.std(ys) / H

    # Average distance to center
    center_x = W / 2
    center_y = H / 2

    dist = np.sqrt((xs - center_x)**2 + (ys - center_y)**2)
    avg_distance_center = np.mean(dist) / 0.5*np.sqrt(W**2 + H**2)

    # Left-right balance
    left_mass = np.sum(xs < center_x)
    right_mass = np.sum(xs >= center_x)

    lr_balance = (left_mass - right_mass) / (left_mass + right_mass)

    # Top-bottom balance
    top_mass = np.sum(ys < center_y)
    bottom_mass = np.sum(ys >= center_y)

    tb_balance = (top_mass - bottom_mass) / (top_mass + bottom_mass)

    return [
        coverage_ratio,
        pixel_std_x,
        pixel_std_y,
        avg_distance_center,
        lr_balance,
        tb_balance
    ]

In [ ]:
coverage_list = []
std_x_list = []
std_y_list = []
center_dist_list = []
lr_balance_list = []
tb_balance_list = []

for participant in valid_participants:
    for condition in ["c1", "c2"]:

        # load image
        image_path = data_path + "p" + str(participant) + "/drawing_data/final_drawing_" + condition + ".png"
        img = cv2.imread(image_path)

        # handle specific case of p20 c2 (erased final drawing -> use intermediate drawing saved in smaller resolution)
        if participant == 20 and condition == "c2":
            img = cv2.resize(img, (1920, 972), interpolation=cv2.INTER_LINEAR)

        # remove blue border
        cropped = img[5:-5, 5:-5]

        features = extract_spatial_features(cropped)

        coverage_list.append(features[0])
        std_x_list.append(features[1])
        std_y_list.append(features[2])
        center_dist_list.append(features[3])
        lr_balance_list.append(features[4])
        tb_balance_list.append(features[5])


features_df["surface_coverage_ratio"] = coverage_list
features_df["pixel_std_x"] = std_x_list
features_df["pixel_std_y"] = std_y_list
features_df["mean_distance_center"] = center_dist_list
features_df["left_right_balance"] = lr_balance_list
features_df["top_bottom_balance"] = tb_balance_list

### Colors analysis

In [ ]:
def extract_color_features(img):

    # remove white background
    mask = ~((img[:,:,0] == 255) &
             (img[:,:,1] == 255) &
             (img[:,:,2] == 255))

    stroke_pixels = img[mask]

    if len(stroke_pixels) == 0:
        return [np.nan]*9

    # number of different colors
    unique_colors = np.unique(stroke_pixels, axis=0)
    n_colors = len(unique_colors)

    # convert stroke pixels to HSV
    hsv_pixels = cv2.cvtColor(stroke_pixels.reshape(-1,1,3), cv2.COLOR_BGR2HSV).reshape(-1,3)

    hue = hsv_pixels[:,0].astype(float)       # [0, 180]
    sat = hsv_pixels[:,1].astype(float) / 255 # [0, 1]
    val = hsv_pixels[:,2].astype(float) / 255 # [0, 1]

    # circular mean for hue
    angles = hue * 2*np.pi / 180

    mean_angle = np.arctan2(np.mean(np.sin(angles)),
                            np.mean(np.cos(angles)))

    avg_hue = (mean_angle * 180 / np.pi) % 180

    std_hue = np.std(hue)

    avg_sat = np.mean(sat)
    std_sat = np.std(sat)

    avg_val = np.mean(val)
    std_val = np.std(val)

    # black vs others
    black_pixels = np.sum((stroke_pixels[:,0] == 0) &
                          (stroke_pixels[:,1] == 0) &
                          (stroke_pixels[:,2] == 0))
    black_ratio = black_pixels / len(stroke_pixels)

    # warm vs cool classification
    # red / orange / yellow → warm
    # green / blue / purple → cool
    # Warm:   0–30  OR  150–179
    # Cool:   30–150
    warm_mask = (hue < 30) | (hue > 150)
    cool_mask = (hue >= 30) & (hue <= 150)

    warm_pixels = np.sum(warm_mask)
    cool_pixels = np.sum(cool_mask)

    warm_cool_ratio = warm_pixels / (cool_pixels + 1e-8)

    return (
        n_colors,
        avg_hue, std_hue,
        avg_sat, std_sat,
        avg_val, std_val,
        black_ratio,
        warm_cool_ratio
    )

In [ ]:
color_count_list = []
avg_hue_list = []
std_hue_list = []
avg_sat_list = []
std_sat_list = []
avg_val_list = []
std_val_list = []
black_ratio_list = []
warm_cool_ratio_list = []

for participant in valid_participants:
    for condition in ["c1", "c2"]:

        # load image
        image_path = data_path + "p" + str(participant) + "/drawing_data/final_drawing_" + condition + ".png"
        img = cv2.imread(image_path)

        # handle specific case of p20 c2 (erased final drawing -> use intermediate drawing saved in smaller resolution)
        if participant == 20 and condition == "c2":
            img = cv2.resize(img, (1920, 972), interpolation=cv2.INTER_LINEAR)

        # remove blue border
        cropped = img[5:-5, 5:-5]

        features = extract_color_features(cropped)

        color_count_list.append(features[0])
        avg_hue_list.append(features[1])
        std_hue_list.append(features[2])
        avg_sat_list.append(features[3])
        std_sat_list.append(features[4])
        avg_val_list.append(features[5])
        std_val_list.append(features[6])
        black_ratio_list.append(features[7])
        warm_cool_ratio_list.append(features[8])


features_df["colors_used_count"] = color_count_list
features_df["mean_hue"] = avg_hue_list
features_df["std_hue"] = std_hue_list
features_df["mean_saturation"] = avg_sat_list
features_df["std_saturation"] = std_sat_list
features_df["mean_brightness"] = avg_val_list
features_df["std_brightness"] = std_val_list
features_df["black_ratio"] = black_ratio_list
features_df["warm_cool_ratio"] = warm_cool_ratio_list

### Drawing sementic content

In [ ]:
# Prompt for the Vision Language Model
VLM_PROMPT_DRAWING = """You are analyzing a drawing created by a human.
Your task is to detect the semantic content of the drawing and output structured features.

Carefully inspect the image and determine whether the following semantic categories are present.

Definitions:
Human: Any depiction of a person, including stick figures, full bodies, or detailed faces representing a person.
Objects: Man-made items such as houses, cars, tools, furniture, buildings, etc.
Nature/animals: Natural elements such as trees, flowers, mountains, sun, clouds, water, or any animals.
Emotional symbols: Symbols that express emotions or feelings, such as hearts, smiley faces, sad faces, tears, broken hearts, etc.
Text indicating names or first person: Written words such as names, “I”, “me”, “my”, signatures, or other identifiable text.
Abstract shapes: Geometric or decorative shapes without clear semantic meaning (circles, triangles, scribbles, repeated patterns, etc.).

For each category, output 1 if present and 0 if absent.

Finally estimate:
intimacy_score: an integer between 1 and 5 representing how intimate and sensitive the topic of the drawing is.

Interpretation of intimacy:
1 – Very impersonal: abstract patterns, random shapes, purely decorative drawings.
2 – Mostly impersonal: landscapes, objects, buildings, neutral scenes.
3 – Moderately personal: scenes with people, animals, or everyday life elements.
4 – Personal/emotional: drawings including emotional symbols, expressive faces, or personal themes.
5 – Highly intimate: drawings expressing strong emotions, personal relationships, self-references, or very personal content.

Important instructions:
Return ONLY valid JSON.
Do not include explanations.
Use only the keys specified below.

Output format:
{
"human": 0 or 1,
"objects": 0 or 1,
"nature_animals": 0 or 1,
"emotional_symbols": 0 or 1,
"text_first_person": 0 or 1,
"abstract_shapes": 0 or 1,
"intimacy_score": integer between 1 and 5
}"""

In [ ]:
def parse_vlm_reply(vlm_reply):
    
    # Remove possible markdown code blocks
    vlm_reply = re.sub(r"```json|```", "", vlm_reply).strip()
    
    # Convert JSON string to Python dict
    data = json.loads(vlm_reply)
    
    return data

In [ ]:
human_presence_list = []
object_presence_list = []
nature_animals_presence_list = []
emotional_symbols_presence_list = []
first_person_text_presence_list = []
abstract_shapes_presence_list = []
VLM_intimacy_score_list = []

for participant in valid_participants:
    for condition in ["c1", "c2"]:

        print("Processing participant:", str(participant), "- condition:", condition)

        # load image
        image_path = data_path + "p" + str(participant) + "/drawing_data/final_drawing_" + condition + ".png"
        img = cv2.imread(image_path)

        # handle specific case of p20 c2 (erased final drawing -> use intermediate drawing saved in smaller resolution)
        if participant == 20 and condition == "c2":
            img = cv2.resize(img, (1920, 972), interpolation=cv2.INTER_LINEAR)

        # remove blue border
        cropped = img[5:-5, 5:-5]

        # Convert the image into a readable format for the vlm
        _, buffer = cv2.imencode(".jpg", cropped)
        drawing_image_base64 = base64.b64encode(buffer).decode("utf-8")

        # Prompt the VLM 3 times and do a majority vote
        human_presence_vote_list = []
        object_presence_vote_list = []
        nature_animals_presence_vote_list = []
        emotional_symbols_presence_vote_list = []
        first_person_text_presence_vote_list = []
        abstract_shapes_presence_vote_list = []
        VLM_intimacy_score_vote_list = []
        for i in range(3):
            conversation_history = [{"role": "system", "content": VLM_PROMPT_DRAWING},
                                    {"role": "user",
                                     "content": "",
                                     "images": [drawing_image_base64]}]
    
            vlm_output = ollama.chat(model="qwen3-vl:235b-a22b-instruct", messages=conversation_history)
            vlm_reply = vlm_output["message"]["content"]
    
            data = parse_vlm_reply(vlm_reply)

            human_presence_vote_list.append(data["human"])
            object_presence_vote_list.append(data["objects"])
            nature_animals_presence_vote_list.append(data["nature_animals"])
            emotional_symbols_presence_vote_list.append(data["emotional_symbols"])
            first_person_text_presence_vote_list.append(data["text_first_person"])
            abstract_shapes_presence_vote_list.append(data["abstract_shapes"])
            VLM_intimacy_score_vote_list.append(data["intimacy_score"])

        # Majority vote (except for the intimacy score estimation for which we just use the mean)
        human_presence_list.append(round(np.mean(human_presence_vote_list)))
        object_presence_list.append(round(np.mean(object_presence_vote_list)))
        nature_animals_presence_list.append(round(np.mean(nature_animals_presence_vote_list)))
        emotional_symbols_presence_list.append(round(np.mean(emotional_symbols_presence_vote_list)))
        first_person_text_presence_list.append(round(np.mean(first_person_text_presence_vote_list)))
        abstract_shapes_presence_list.append(round(np.mean(abstract_shapes_presence_vote_list)))
        VLM_intimacy_score_list.append(np.mean(VLM_intimacy_score_vote_list))


features_df["drawing_human_presence"] = human_presence_list
features_df["drawing_object_presence"] = object_presence_list
features_df["drawing_nature_animals_presence"] = nature_animals_presence_list
features_df["drawing_emotional_symbols_presence"] = emotional_symbols_presence_list
features_df["drawing_first_person_text_presence"] = first_person_text_presence_list
features_df["drawing_abstract_shapes_presence"] = abstract_shapes_presence_list
features_df["drawing_VLM_intimacy_score"] = VLM_intimacy_score_list

## Speech dynamics features

In [ ]:
# Initialize openSMILE with eGeMAPS functional features
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)

features_of_interest = [
    "F0semitoneFrom27.5Hz_sma3nz_amean",
    "F0semitoneFrom27.5Hz_sma3nz_stddevNorm",
    "F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2",
    "loudness_sma3_amean",
    "loudness_sma3_stddevNorm",
    "jitterLocal_sma3nz_amean",
    "shimmerLocaldB_sma3nz_amean",
    "HNRdBACF_sma3nz_amean",
]

In [ ]:
# Load Silero VAD
model, utils = torch.hub.load(
    repo_or_dir='snakers4/silero-vad',
    model='silero_vad',
    force_reload=False
)

(get_speech_timestamps,
 save_audio,
 read_audio,
 VADIterator,
 collect_chunks) = utils


def compute_speech_time(audio_path):
    wav, sr = torchaudio.load(audio_path)

    # Ensure mono
    if wav.shape[0] > 1:
        wav = torch.mean(wav, dim=0, keepdim=True)

    # Resample from 44.1 KHz to 16 KHz
    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)
        sr = 16000

    speech_timestamps = get_speech_timestamps(wav.squeeze(0), model, sampling_rate=sr)

    total_speech_time = sum((ts['end'] - ts['start']) for ts in speech_timestamps) / sr

    return total_speech_time

In [ ]:
# Create storage dictionary
opensmile_data = {feat: [] for feat in features_of_interest}
speech_time_list = []

for participant in valid_participants:
    for condition in ["c1", "c2"]:

        audio_file_path = data_path + "p" + str(participant) + "/audio_data/p" + str(participant) + condition + "_copy.wav"

        # Extract opensmile features
        print("Processing participant:", str(participant), "- condition:", condition)
        result = smile.process_file(audio_file_path)

        # opensmile result is a pandas dataframe (1 row for functionals)
        for feature in features_of_interest:
            value = result[feature].values[0]
            opensmile_data[feature].append(value)

        # Compute speech time
        speech_time = compute_speech_time(audio_file_path)
        speech_time_list.append(speech_time)

# Add opensmile features to dataframe
for feature in features_of_interest:
    features_df[feature] = opensmile_data[feature]

# Add speech time to dataframe
features_df["total_speech_time"] = speech_time_list

# Add speech ratio to dataframe
features_df["speech_ratio"] = (features_df["total_speech_time"] / features_df["activity_time"])

## Speech production features (transcript)

### Word count

In [ ]:
word_count_list = []
mean_word_count_sentence_list = []

for participant in valid_participants:
    for condition in ["c1", "c2"]:
        transcript_file_path = data_path + "p" + str(participant) + "/audio_data/p" + str(participant) + condition + ".txt"
        with open(transcript_file_path, "r") as file:
            text = file.read()

        # Compute number of words
        words = re.findall(r"\b\w+\b", text)
        total_words = len(words)
    
        # Compute number of words per sentence
        sentences = re.split(r"[.!?]+", text)
        sentences = [s.strip() for s in sentences if s.strip()]
        num_sentences = len(sentences)
        avg_words_per_sentence = total_words / num_sentences if num_sentences > 0 else 0

        word_count_list.append(total_words)
        mean_word_count_sentence_list.append(avg_words_per_sentence)

features_df["word_count"] = word_count_list
features_df["mean_word_count_sentence"] = mean_word_count_sentence_list # NEED DOUBLE CHECKING
features_df["word_count_per_sec"] = features_df["word_count"] / features_df["activity_time"]

### Speech disfluencies

In [ ]:
hesitation_markers_ratio_list = []

for participant in valid_participants:
    for condition in ["c1", "c2"]:
        transcript_file_path = data_path + "p" + str(participant) + "/audio_data/p" + str(participant) + condition + ".txt"
        with open(transcript_file_path, "r") as file:
            text = file.read()

        # Compute number of words
        words = re.findall(r"\b\w+\b", text.lower())
        total_words = len(words)
    
        # Compute number of hesitation markers
        hesitation_markers = ["um", "uh"]
        hesitation_count = sum(words.count(h) for h in hesitation_markers)

        hesitation_markers_ratio_list.append(hesitation_count / total_words)

features_df["hesitation_markers_ratio"] = hesitation_markers_ratio_list

### Lexical diversity (Moving Average Type-Token Ratio, MATTR)

In [ ]:
def mattr(words, window_size=50):

    if len(words) < window_size:
        return len(set(words)) / len(words) if len(words) > 0 else 0

    ttr_values = []

    for i in range(len(words) - window_size + 1):
        window = words[i:i+window_size]
        ttr = len(set(window)) / window_size
        ttr_values.append(ttr)

    return np.mean(ttr_values)

In [ ]:
mattr_list = []

for participant in valid_participants:
    for condition in ["c1", "c2"]:
        transcript_file_path = data_path + "p" + str(participant) + "/audio_data/p" + str(participant) + condition + ".txt"
        with open(transcript_file_path, "r") as file:
            text = file.read()

        # Compute number of words
        words = re.findall(r"\b\w+\b", text.lower())
        mattr_list.append(mattr(words))

features_df["mattr"] = mattr_list

### Self-disclosure depth (Altman and Taylor framework)

In [ ]:
VLM_PROMPT_TRANSCRIPT = """
You are annotating a transcript for psychological research.

Your task is to rate the level of SELF-DISCLOSURE in a single utterance
based on Social Penetration Theory (Altman & Taylor, 1974).

Self-disclosure refers to when a speaker reveals personal information
about themselves, their feelings, experiences, attitudes, or opinions.

Use the following scale:

0 = No self-disclosure
The utterance does not reveal personal information about the speaker.
Examples:
- Describing objects or drawings
- Talking about the task
- Neutral observations
- Questions
Example: "There is a tree next to the house."

1 = Peripheral layer (outer layer)
Superficial or public personal information.
Simple preferences, mild opinions, or basic self-references.
Examples:
- "I like drawing trees."
- "I usually prefer blue."

2 = Intermediate layer (middle layer)
More personal attitudes, opinions, or experiences.
The speaker reveals personal evaluations or experiences.
Examples:
- "I used to draw a lot when I was younger."
- "I think art is really relaxing for me."

3 = Intimate layer (core layer)
Deeply personal or private information about the speaker.
Strong emotions, personal struggles, or meaningful personal experiences.
Examples:
- "Drawing helped me when I was feeling very lonely."
- "This reminds me of difficult times in my childhood."

Important instructions:

Return ONLY one integer: 0, 1, 2, or 3.
Do NOT explain your answer.
Do NOT output anything else.
"""

In [ ]:
transcript_mean_disclosure_depth_list = []

for participant in valid_participants:
    for condition in ["c1", "c2"]:

        print("Processing participant:", str(participant), "- condition:", condition)

        # Extract full transcript
        transcript_file_path = data_path + "p" + str(participant) + "/audio_data/p" + str(participant) + condition + ".txt"
        with open(transcript_file_path, "r") as file:
            text = file.read()

        # Divide transcript into utterances
        utterances = re.findall(r'[^.!?]+[.!?]', text)
        
        # Clean empty segments
        utterances = [u.strip() for u in utterances if u.strip()]

        print("Number of utterances to process: ", str(len(utterances)))
        
        utterances_disclosure_depth_list = []
        utterance_results = []
        conversation_history = [{"role": "system", "content": VLM_PROMPT_TRANSCRIPT}]
        for i, utterance in enumerate(utterances):
            print(i)
            conversation_history.append({"role": "user", "content": utterance})

            # Do it 3 times per utterance and do a majority vote
            scores = []
            for _ in range(3):
                vlm_output = ollama.chat(model="qwen3-vl:235b-a22b-instruct", messages=conversation_history)
                vlm_reply = vlm_output["message"]["content"].strip()

                scores.append(int(vlm_reply))

            if len(set(scores)) == 3:
                # All scores different → use median
                self_disclosure_depth_result_vote = int(np.median(scores))
            else:
                # Otherwise use majority vote
                self_disclosure_depth_result_vote = Counter(scores).most_common(1)[0][0]
                
            utterances_disclosure_depth_list.append(self_disclosure_depth_result_vote)
            utterance_results.append({"utterance_id": i, "disclosure_depth_score": self_disclosure_depth_result_vote, "utterance": utterance})

            conversation_history.append({"role": "assistant", "content": str(self_disclosure_depth_result_vote)})

        # Final participant intimacy score is the mean intimacy score of all utterances
        transcript_disclosure_depth_score = np.mean(utterances_disclosure_depth_list)
        transcript_mean_disclosure_depth_list.append(transcript_disclosure_depth_score)

        # Save utterance table
        utterance_df = pd.DataFrame(utterance_results)
        output_file = data_path + "p" + str(participant) + "/audio_data/p" + str(participant) + condition + "_utterance_scores.xlsx"
        utterance_df.to_excel(output_file, index=False)

features_df["mean_disclosure_depth"] = transcript_mean_disclosure_depth_list

## Display and save final features dataframe

In [ ]:
features_df.to_excel("../extracted_features/extracted_features_final.xlsx", index=False)

features_df